## GSAT trend patterns

In [ ]:
# In[1]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
# %%
# define function
import src.SAT_function_Obs_Fingerprint as data_process
import src.Data_Preprocess as preprocess
from src.Statistic_cal import pattern_rmse

In [ ]:
# import src.slurm_cluster as scluster
# client, scluster = scluster.init_dask_slurm_cluster()

In [ ]:
def func_mk(x):
    """
    Mann-Kendall test for trend
    """
    results = data_process.apply_mannkendall(x)
    slope = results[0]
    p_val = results[1]
    return slope, p_val

In [ ]:
# Input the observational trend
variable_name = ['2013-2022', '1993-2022', '1963-2022', '1979-2022']

# Input the Observational forced trend (wrt MMEM GSAT)
dir_forced_input = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIGS5_S6/trend_forced_HadCRUT5_annual/'

HadCRUT5_forced_trend_da = {}
HadCRUT5_p_val_da = {}

for interval in variable_name:
    HadCRUT5_forced_trend_da[interval] = xr.open_dataset(dir_forced_input + 'forced_HadCRUT5_MMLE_MK_trend_1950-2022_sliding.nc').trend.sel(period=interval)
    HadCRUT5_p_val_da[interval] = xr.open_dataset(dir_forced_input + 'forced_HadCRUT5_MMLE_MK_trend_1950-2022_sliding.nc').p_value.sel(period=interval)

In [ ]:
HadCRUT5_forced_trend_da

In [ ]:
# Input the MMEM annual trend
dir_model_in = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/{model}/SMILE_forced/'
model_name = ["MMLE", "MIROC6", "MPI_ESM", "ACCESS", "EC_Earth3", "IPSL_CM6A", "CESM2", "CanESM5"]

LE_forced_trend_da = {}
LE_forced_p_val_da = {}

for model in model_name:
    LE_forced_trend_da[model] = {}
    LE_forced_p_val_da[model] = {}
    for interval in variable_name:
        LE_forced_trend_da[model][interval] = xr.open_dataset(dir_model_in.format(model=model) + '{model}_ENSmean_forced_MK_trend_1950-2022_sliding.nc'.format(model=model)).trend.sel(period=interval)
        LE_forced_p_val_da[model][interval] = xr.open_dataset(dir_model_in.format(model=model) + '{model}_ENSmean_forced_MK_trend_1950-2022_sliding.nc'.format(model=model)).p_value.sel(period=interval)

In [ ]:
# pattern difference between LE forced trend and obs trend
LE_forced_trend_diff_da = {}
for model in model_name:
    dir_diff_trend = f'/work/mh0033/m301036/OBS_LPS_revision/script/FIG3/Pattern_diff/{model}'
    LE_forced_trend_diff_da[model] = {}
    for interval in variable_name:
        if model == 'MMLE':
            LE_forced_trend_diff_da[model][interval] = xr.open_dataset(f'/work/mh0033/m301036/OBS_LPS_revision/script/FIG3/Pattern_diff/{model}_OBS_forced_pattern_diff_1950_2022.nc').trend_diff.sel(period=interval)
        else:
            LE_forced_trend_diff_da[model][interval] = xr.open_dataset(f'{dir_diff_trend}/{model}_OBS_forced_pattern_diff_1950_2022.nc').trend_diff.sel(period=interval)


In [ ]:
# check the min and max value of the trend
# HadCRUT5
for interval in variable_name:
    print('HadCRUT5', interval, 'min:', HadCRUT5_forced_trend_da[interval].min().values, 'max:', HadCRUT5_forced_trend_da[interval].max().values)

### The amplitude ratio is 
$\text{bias\_fraction} = \dfrac{A_{\text{bias}}}{A_{\text{obs}}}
 = \dfrac{\mathrm{RMS}(F_{\text{ENS}} - F_{\text{OBS}})}{\mathrm{RMS}(F_{\text{OBS}})}$.

In [ ]:
def area_weighted_rms(field, lat):
    w = np.cos(np.deg2rad(lat))
    w = w / w.mean()
    w2 = w.broadcast_like(field)
    wsum = w2.sum(dim=("lat", "lon"))
    return np.sqrt((w2 * field**2).sum(dim=("lat", "lon")) / wsum)


In [ ]:
LE_forced_trend_diff_amplitude_da = {}
for model in model_name:
    LE_forced_trend_diff_amplitude_da[model] = {}
    for interval in variable_name:
        LE_forced_trend_diff_amplitude_da[model][interval] = area_weighted_rms(LE_forced_trend_diff_da[model][interval], LE_forced_trend_diff_da[model][interval].lat)

In [ ]:
LE_forced_trend_diff_amplitude_da

In [ ]:
# calculate the observed forced trend amplitude
HadCRUT5_forced_trend_amplitude_da = {}
for interval in variable_name:
    HadCRUT5_forced_trend_amplitude_da[interval] = area_weighted_rms(HadCRUT5_forced_trend_da[interval], HadCRUT5_forced_trend_da[interval].lat)

In [ ]:
HadCRUT5_forced_trend_amplitude_da

In [ ]:
# calculate the bias fraction
LE_forced_trend_diff_fraction_da = {}
for model in model_name:
    LE_forced_trend_diff_fraction_da[model] = {}
    for interval in variable_name:
        LE_forced_trend_diff_fraction_da[model][interval] = LE_forced_trend_diff_amplitude_da[model][interval] / HadCRUT5_forced_trend_amplitude_da[interval]*100

In [ ]:
LE_forced_trend_diff_fraction_da["IPSL_CM6A"]

### Plotting with the Robinson Projections

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.colors import BoundaryNorm, ListedColormap
from src.plot_func import *
set_science_advances_style(column='double')  # or 'single'

def plot_trend_with_significance(trend_data, lats, lons, p_values, GMST_p_values=None, levels=None, extend=None, cmap=None, 
                                 title="", ax=None, show_xticks=False, show_yticks=False):
    """
    Plot the trend spatial pattern using Robinson projection with significance overlaid.

    Parameters:
    - trend_data: 2D numpy array with the trend values.
    - lats, lons: 1D arrays of latitudes and longitudes.
    - p_values: 2D array with p-values for each grid point.
    - GMST_p_values: 2D array with GMST p-values for each grid point.
    - title: Title for the plot.
    - ax: Existing axis to plot on. If None, a new axis will be created.
    - show_xticks, show_yticks: Boolean flags to show x and y axis ticks.
    
    Returns:
    - contour_obj: The contour object from the plot.
    """

    # Create a new figure/axis if none is provided
    if ax is None:
        fig, ax = plt.subplots(figsize=(20, 15), subplot_kw={'projection': ccrs.Robinson()})
        ax.set_global()
  
    insignificance_mask = p_values >= 0.05
    
    # Plotting
    # contour_obj = ax.pcolormesh(lons, lats, trend_data,  cmap='RdBu_r',vmin=-5.0, vmax=5.0, transform=ccrs.PlateCarree(central_longitude=180), shading='auto')
    contour_obj = ax.contourf(lons, lats, trend_data, levels=levels, extend=extend, cmap=cmap, transform=ccrs.PlateCarree(central_longitude=0))

    # Plot significance masks with different hatches
    ax.contourf(lons, lats, insignificance_mask, levels=[0, 0.05, 1.0],hatches=[None,'///'], colors='none', transform=ccrs.PlateCarree())

    ax.coastlines(resolution='110m')
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                      color='gray', alpha=0.35, linestyle='--')

    # Disable labels on the top and right of the plot
    gl.top_labels = False
    gl.right_labels = False

    # Enable labels on the bottom and left of the plot
    gl.bottom_labels = show_xticks
    gl.left_labels = show_yticks
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    gl.xlabel_style = {'size': 15}
    gl.ylabel_style = {'size': 15}
    
    if show_xticks:
        gl.bottom_labels = True
    if show_yticks:
        gl.left_labels = True
    
    ax.set_title(title, loc='center', fontsize=18, pad=5.0)

    return contour_obj

In [ ]:
# define an asymmetric colormap
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.colors import BoundaryNorm
import cartopy.util as cutil
import seaborn as sns
import matplotlib.colors as mcolors
import palettable

intervals = [-0.2, -0.15, -0.1, -0.05, 0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.9, 1.1, 1.3]

# Normalizing the intervals to [0, 1]
min_interval = min(intervals)
max_interval = max(intervals)
normalized_intervals = [(val - min_interval) / (max_interval - min_interval) for val in intervals]

cmap=mcolors.ListedColormap(palettable.cmocean.diverging.Balance_20.mpl_colors)

### Plot the Original, Forced, MMEM trend patterns

In [ ]:
# # arange data into a list arrocding to the variable name
# trend_2013_2022 = {"HadCRUT5":HadCRUT5_forced_trend_da['2013-2022'], 
#             "MMLE":LE_forced_trend_da['MMLE']['2013-2022'],
#            "MIROC6":LE_forced_trend_da['MIROC6']['2013-2022'],
#            "MPI_ESM":LE_forced_trend_da['MPI_ESM']['2013-2022'],
#            "ACCESS":LE_forced_trend_da['ACCESS']['2013-2022'], 
#            "EC_Earth3":LE_forced_trend_da['EC_Earth3']['2013-2022'],
#            "IPSL_CM6A":LE_forced_trend_da['IPSL_CM6A']['2013-2022'],
#            "CESM2":LE_forced_trend_da['CESM2']['2013-2022'],
#            "CanESM5":LE_forced_trend_da['CanESM5']['2013-2022']
#             }

# p_value_2013_2022 = {"HadCRUT5":HadCRUT5_p_val_da['2013-2022'],
#                 "MMLE":LE_forced_p_val_da['MMLE']['2013-2022'],
#                 "MIROC6":LE_forced_p_val_da['MIROC6']['2013-2022'],
#                 "MPI_ESM":LE_forced_p_val_da['MPI_ESM']['2013-2022'],
#                 "ACCESS":LE_forced_p_val_da['ACCESS']['2013-2022'],
#                 "EC_Earth3":LE_forced_p_val_da['EC_Earth3']['2013-2022'],
#                 "IPSL_CM6A":LE_forced_p_val_da['IPSL_CM6A']['2013-2022'],
#                 "CESM2":LE_forced_p_val_da['CESM2']['2013-2022'],
#                 "CanESM5":LE_forced_p_val_da['CanESM5']['2013-2022']
#                 }
# trend_1993_2022 = {"HadCRUT5":HadCRUT5_forced_trend_da['1993-2022'],
#             "MMLE":LE_forced_trend_da['MMLE']['1993-2022'],
#             "MIROC6":LE_forced_trend_da['MIROC6']['1993-2022'],
#             "MPI_ESM":LE_forced_trend_da['MPI_ESM']['1993-2022'],
#             "ACCESS":LE_forced_trend_da['ACCESS']['1993-2022'],
#             "EC_Earth3":LE_forced_trend_da['EC_Earth3']['1993-2022'],
#             "IPSL_CM6A":LE_forced_trend_da['IPSL_CM6A']['1993-2022'],
#             "CESM2":LE_forced_trend_da['CESM2']['1993-2022'],
#             "CanESM5":LE_forced_trend_da['CanESM5']['1993-2022']
#             }
# p_value_1993_2022 = {"HadCRUT5":HadCRUT5_p_val_da['1993-2022'],
#                 "MMLE":LE_forced_p_val_da['MMLE']['1993-2022'],
#                 "MIROC6":LE_forced_p_val_da['MIROC6']['1993-2022'],
#                 "MPI_ESM":LE_forced_p_val_da['MPI_ESM']['1993-2022'],
#                 "ACCESS":LE_forced_p_val_da['ACCESS']['1993-2022'],
#                 "EC_Earth3":LE_forced_p_val_da['EC_Earth3']['1993-2022'],
#                 "IPSL_CM6A":LE_forced_p_val_da['IPSL_CM6A']['1993-2022'],
#                 "CESM2":LE_forced_p_val_da['CESM2']['1993-2022'],
#                 "CanESM5":LE_forced_p_val_da['CanESM5']['1993-2022']
#                 }
# trend_1963_2022 = {"HadCRUT5":HadCRUT5_forced_trend_da['1963-2022'],
#             "MMLE":LE_forced_trend_da['MMLE']['1963-2022'],
#             "MIROC6":LE_forced_trend_da['MIROC6']['1963-2022'],
#             "MPI_ESM":LE_forced_trend_da['MPI_ESM']['1963-2022'],
#             "ACCESS":LE_forced_trend_da['ACCESS']['1963-2022'],
#             "EC_Earth3":LE_forced_trend_da['EC_Earth3']['1963-2022'],
#             "IPSL_CM6A":LE_forced_trend_da['IPSL_CM6A']['1963-2022'],
#             "CESM2":LE_forced_trend_da['CESM2']['1963-2022'],
#             "CanESM5":LE_forced_trend_da['CanESM5']['1963-2022']
#             }
# p_value_1963_2022 = {"HadCRUT5":HadCRUT5_p_val_da['1963-2022'],
#                 "MMLE":LE_forced_p_val_da['MMLE']['1963-2022'],
#                 "MIROC6":LE_forced_p_val_da['MIROC6']['1963-2022'],
#                 "MPI_ESM":LE_forced_p_val_da['MPI_ESM']['1963-2022'],
#                 "ACCESS":LE_forced_p_val_da['ACCESS']['1963-2022'],
#                 "EC_Earth3":LE_forced_p_val_da['EC_Earth3']['1963-2022'],
#                 "IPSL_CM6A":LE_forced_p_val_da['IPSL_CM6A']['1963-2022'],
#                 "CESM2":LE_forced_p_val_da['CESM2']['1963-2022'],
#                 "CanESM5":LE_forced_p_val_da['CanESM5']['1963-2022']
#                 }
# trend_1979_2022 = {"HadCRUT5":HadCRUT5_forced_trend_da['1979-2022'],
#             "MMLE":LE_forced_trend_da['MMLE']['1979-2022'],
#             "MIROC6":LE_forced_trend_da['MIROC6']['1979-2022'],
#             "MPI_ESM":LE_forced_trend_da['MPI_ESM']['1979-2022'],
#             "ACCESS":LE_forced_trend_da['ACCESS']['1979-2022'],
#             "EC_Earth3":LE_forced_trend_da['EC_Earth3']['1979-2022'],
#             "IPSL_CM6A":LE_forced_trend_da['IPSL_CM6A']['1979-2022'],
#             "CESM2":LE_forced_trend_da['CESM2']['1979-2022'],
#             "CanESM5":LE_forced_trend_da['CanESM5']['1979-2022']
#             }
# p_value_1979_2022 = {"HadCRUT5":HadCRUT5_p_val_da['1979-2022'],
#                 "MMLE":LE_forced_p_val_da['MMLE']['1979-2022'],
#                 "MIROC6":LE_forced_p_val_da['MIROC6']['1979-2022'],
#                 "MPI_ESM":LE_forced_p_val_da['MPI_ESM']['1979-2022'],
#                 "ACCESS":LE_forced_p_val_da['ACCESS']['1979-2022'],
#                 "EC_Earth3":LE_forced_p_val_da['EC_Earth3']['1979-2022'],
#                 "IPSL_CM6A":LE_forced_p_val_da['IPSL_CM6A']['1979-2022'],
#                 "CESM2":LE_forced_p_val_da['CESM2']['1979-2022'],
#                 "CanESM5":LE_forced_p_val_da['CanESM5']['1979-2022']
#                 }   

In [ ]:
# trend_2013_2022

In [ ]:
# arange data into a list arrocding to the variable name
trend_2013_2022 = {"HadCRUT5":HadCRUT5_forced_trend_da['2013-2022'], 
            "MMLE":LE_forced_trend_diff_da['MMLE']['2013-2022'],
           "MIROC6":LE_forced_trend_diff_da['MIROC6']['2013-2022'],
           "MPI_ESM":LE_forced_trend_diff_da['MPI_ESM']['2013-2022'],
           "ACCESS":LE_forced_trend_diff_da['ACCESS']['2013-2022'], 
           "EC_Earth3":LE_forced_trend_diff_da['EC_Earth3']['2013-2022'],
           "IPSL_CM6A":LE_forced_trend_diff_da['IPSL_CM6A']['2013-2022'],
           "CESM2":LE_forced_trend_diff_da['CESM2']['2013-2022'],
           "CanESM5":LE_forced_trend_diff_da['CanESM5']['2013-2022']
            }
            
trend_1993_2022 = {"HadCRUT5":HadCRUT5_forced_trend_da['1993-2022'],
            "MMLE":LE_forced_trend_diff_da['MMLE']['1993-2022'],
            "MIROC6":LE_forced_trend_diff_da['MIROC6']['1993-2022'],
            "MPI_ESM":LE_forced_trend_diff_da['MPI_ESM']['1993-2022'],
            "ACCESS":LE_forced_trend_diff_da['ACCESS']['1993-2022'],
            "EC_Earth3":LE_forced_trend_diff_da['EC_Earth3']['1993-2022'],
            "IPSL_CM6A":LE_forced_trend_diff_da['IPSL_CM6A']['1993-2022'],
            "CESM2":LE_forced_trend_diff_da['CESM2']['1993-2022'],
            "CanESM5":LE_forced_trend_diff_da['CanESM5']['1993-2022']
            }

trend_1963_2022 = {"HadCRUT5":HadCRUT5_forced_trend_da['1963-2022'],
            "MMLE":LE_forced_trend_diff_da['MMLE']['1963-2022'],
            "MIROC6":LE_forced_trend_diff_da['MIROC6']['1963-2022'],
            "MPI_ESM":LE_forced_trend_diff_da['MPI_ESM']['1963-2022'],
            "ACCESS":LE_forced_trend_diff_da['ACCESS']['1963-2022'],
            "EC_Earth3":LE_forced_trend_diff_da['EC_Earth3']['1963-2022'],
            "IPSL_CM6A":LE_forced_trend_diff_da['IPSL_CM6A']['1963-2022'],
            "CESM2":LE_forced_trend_diff_da['CESM2']['1963-2022'],
            "CanESM5":LE_forced_trend_diff_da['CanESM5']['1963-2022']
            }
trend_1979_2022 = {"HadCRUT5":HadCRUT5_forced_trend_da['1979-2022'],
            "MMLE":LE_forced_trend_diff_da['MMLE']['1979-2022'],
            "MIROC6":LE_forced_trend_diff_da['MIROC6']['1979-2022'],
            "MPI_ESM":LE_forced_trend_diff_da['MPI_ESM']['1979-2022'],
            "ACCESS":LE_forced_trend_diff_da['ACCESS']['1979-2022'],
            "EC_Earth3":LE_forced_trend_diff_da['EC_Earth3']['1979-2022'],
            "IPSL_CM6A":LE_forced_trend_diff_da['IPSL_CM6A']['1979-2022'],
            "CESM2":LE_forced_trend_diff_da['CESM2']['1979-2022'],
            "CanESM5":LE_forced_trend_diff_da['CanESM5']['1979-2022']
            }

In [ ]:
# define an asymmetric colormap
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.colors import BoundaryNorm
import cartopy.util as cutil
import seaborn as sns
import matplotlib.colors as mcolors
import palettable

In [ ]:
# # pattern correlation betwenn observed forced pattern vs. Model simulated forced pattern
# import scipy.stats as stats

# trend_pattern_correlation_10yr = []

# for i in range(8):
#     trend_pattern_correlation_10yr.append(stats.pearsonr(trend_2013_2022['HadCRUT5'].values.flatten(), trend_2013_2022[model_name[i]].values.flatten())[0])

# trend_pattern_correlation_10yr 

In [ ]:
# trend_pattern_correlation_30yr = []
# for i in range(len(model_name)):
#     trend_pattern_correlation_30yr.append(stats.pearsonr(trend_1993_2022['HadCRUT5'].values.flatten(), trend_1993_2022[model_name[i]].values.flatten())[0])
# trend_pattern_correlation_30yr

In [ ]:
# trend_pattern_correlation_44yr = []
# for i in range(len(model_name)):
#     trend_pattern_correlation_44yr.append(stats.pearsonr(trend_1979_2022['HadCRUT5'].values.flatten(), trend_1979_2022[model_name[i]].values.flatten())[0])
# trend_pattern_correlation_44yr

In [ ]:
# trend_pattern_correlation_60yr = []
# for i in range(len(model_name)):
#     trend_pattern_correlation_60yr.append(stats.pearsonr(trend_1963_2022['HadCRUT5'].values.flatten(), trend_1963_2022[model_name[i]].values.flatten())[0])
# trend_pattern_correlation_60yr

In [ ]:
# # save the pattern correlation from the second to the last, which corresponds to the MMEM, CanESM5, IPSL, EC-Earth3, ACCESS, MPI-ESM, MIROC6
# with open('pattern_correlations_ENS_Obs_Forced.txt', 'w') as file:
#     file.write('10-year Trend Pattern Correlations:\n')
#     for correlation in trend_pattern_correlation_10yr:
#         file.write(f"{correlation}\n")

#     file.write('\n30-year Trend Pattern Correlations:\n')
#     for correlation in trend_pattern_correlation_30yr:
#         file.write(f"{correlation}\n")

#     file.write('\n44-year Trend Pattern Correlations:\n')
#     for correlation in trend_pattern_correlation_44yr:
#         file.write(f"{correlation}\n")
        
#     file.write('\n60-year Trend Pattern Correlations:\n')
#     for correlation in trend_pattern_correlation_60yr:
#         file.write(f"{correlation}\n")

In [ ]:
# Plotting
lat = trend_2013_2022['HadCRUT5'].lat
lon = trend_2013_2022['HadCRUT5'].lon
lat, lon 

titles_rows = ["HadCRUT5", "MMLE", "MIROC6", "MPI-ESM1.2-LR","ACCESS-ESM1.5", "EC-Earth3", "IPSL-CM6A-LR", "CESM2", "CanESM5"]
# rows_label = ["a", "b", "c", "d", "e", "f", "g"]
rows_label = ["A", "B", "C", "D", "E", "F", "G", "H", "I"]
titles_columns = ["2013-2022 (10yr)", "1993-2022 (30yr)", "1963-2022 (60yr)"]
# ["10-year (2013-2022)", "30-year (1993-2022)", "60-year (1963-2022)"]
import cartopy.util as cutil
import seaborn as sns
import matplotlib.colors as mcolors
import palettable

periods = ["10yr", "30yr", "60yr"]
variable_name = ["HadCRUT5", "MMLE", "MIROC6", "MPI_ESM", "ACCESS", "EC_Earth3", "IPSL_CM6A", "CESM2", "CanESM5"]

intervals = np.arange(-1.0, 1.1, 0.1)
intervals_diff = np.arange(-0.5, 0.55, 0.05)
n_bins = len(intervals_diff) - 1

norm_forced = BoundaryNorm(boundaries=intervals_diff, ncolors=n_bins)

cmap = mcolors.ListedColormap(palettable.cmocean.diverging.Balance_20.mpl_colors)
extend = 'both'
panel_fs = 10
row_label_fs = 8
corr_fs = 9
cbar_fs = 8
# set_science_advances_style(
fig = plt.figure(figsize=(20, 35)) 
gs = gridspec.GridSpec(9, 3, figure=fig, wspace=0.05, hspace=0.05)

# Create a 8x3 grid of subplots
axes = {}
obs_contour = None
diff_contour = None
for i, var in enumerate(variable_name):
    for j, period in enumerate(periods):
        is_left = j == 0
        is_bottom_row = i >= 8
        
        ax = plt.subplot(gs[i, j], projection=ccrs.Robinson(180))
        ax.set_global()
        axes[i, j] = ax
        if j == 0:
            trend_data = trend_2013_2022[var]
        elif j == 1:
            trend_data = trend_1993_2022[var]
        else:
            trend_data = trend_1963_2022[var]

        axis_lon = trend_data.get_axis_num("lon")
        trend_with_cyclic, lons_cyclic = cutil.add_cyclic_point(trend_data, coord=lon, axis=axis_lon)

        if i == 0:
            contour = plot_data(
                trend_with_cyclic, lat, lons_cyclic,
                levels=intervals, extend="both",
                cmap=cmap, title=" ", ax=ax,
                show_xticks=is_bottom_row, show_yticks=is_left
            )
            if obs_contour is None:
                obs_contour = contour
        else:
            contour = plot_data(
                trend_with_cyclic, lat, lons_cyclic,
                levels=intervals_diff, extend="both", norm=norm_forced,
                cmap=cmap, title=" ", ax=ax,
                show_xticks=is_bottom_row, show_yticks=is_left
            )
            if diff_contour is None:
                diff_contour = contour

# add the title for each row
for i, var in enumerate(variable_name):
    axes[i, 0].text(-0.2, 0.5, titles_rows[i], va='center', ha='center', rotation=90, 
                    fontsize=20,transform=axes[i, 0].transAxes)
    axes[i, 0].text(-0.08, 1.05, rows_label[i], va='bottom', ha='right', rotation='horizontal',
                    fontsize=24, fontweight='bold', transform=axes[i, 0].transAxes)

# add the title for each column
for j in range(3):
    axes[0,j].text(0.5, 1.05, titles_columns[j], va='bottom', ha='center', rotation='horizontal', 
                   fontsize=20,transform=axes[0, j].transAxes)

# Add the Bias fraction text
# arange model according to the variable name
model_name = ["MMLE", "MIROC6", "MPI_ESM", "ACCESS", "EC_Earth3", "IPSL_CM6A", "CESM2", "CanESM5"]
for i, model in enumerate(model_name):
    axes[i+1, 0].text(0.35, 1.0, f"bias_frac: {LE_forced_trend_diff_fraction_da[model]['2013-2022']:.2f}"+"%", va='bottom', ha='left', fontsize=20,transform=axes[i+1, 0].transAxes)
    axes[i+1, 1].text(0.35, 1.0, f"bias_frac: {LE_forced_trend_diff_fraction_da[model]['1993-2022']:.2f}"+"%", va='bottom', ha='left', fontsize=20,transform=axes[i+1, 1].transAxes)
    axes[i+1, 2].text(0.35, 1.0, f"bias_frac: {LE_forced_trend_diff_fraction_da[model]['1963-2022']:.2f}"+"%", va='bottom', ha='left', fontsize=20,transform=axes[i+1, 2].transAxes)

# Add horizontal colorbars
cbar_ax = fig.add_axes([0.1, 0.08, 0.35, 0.01])
cbar = plt.colorbar(obs_contour, cax=cbar_ax, orientation='horizontal', extend="both")
cbar.ax.tick_params(labelsize=16)
cbar.set_label('Observation SAT trend (°C per decade)', fontsize=18)
cbar.ax.tick_params(direction='out', length=8, width=2)  
# remove the minor ticks from the bar labels
cbar.ax.minorticks_off()

cbar_ax_diff = fig.add_axes([0.55, 0.08, 0.35, 0.01])
cbar_diff = plt.colorbar(diff_contour, cax=cbar_ax_diff, orientation='horizontal', extend="both")
cbar_diff.ax.tick_params(labelsize=16)
cbar_diff.set_label('Model bias SAT trend difference (°C per decade)', fontsize=16)
cbar_diff.ax.tick_params(direction='out', length=8, width=2)
cbar_diff.ax.minorticks_off()

# plt.figure(constrained_layout=True)
figure_output = '/work/mh0033/m301036/OBS_LPS_revision/docs/Figs/FIG_S7_S8/'
for ext in ("png", "pdf",):
    fig.savefig(figure_output+f"FIGURE_S7_OBS_LE_forced_trend_diff_bias_fraction.{ext}", format=ext, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Plotting
lat = trend_2013_2022['HadCRUT5'].lat
lon = trend_2013_2022['HadCRUT5'].lon
lat, lon 

titles_rows = ["HadCRUT5", "MMLE", "MIROC6", "MPI-ESM1.2-LR","ACCESS-ESM1.5", "EC-Earth3", "IPSL-CM6A-LR", "CESM2", "CanESM5"]
# rows_label = ["a", "b", "c", "d", "e", "f", "g"]
rows_label = ["A", "B", "C", "D", "E", "F", "G", "H", "I"]
titles_columns = ["2013-2022 (10yr)", "1993-2022 (30yr)", "1979-2022 (44yr)", "1963-2022 (60yr)"]
# ["10-year (2013-2022)", "30-year (1993-2022)", "60-year (1963-2022)"]
import cartopy.util as cutil
import seaborn as sns
import matplotlib.colors as mcolors
import palettable

periods = ["10yr", "30yr", "44yr", "60yr"]
variable_name = ["HadCRUT5", "MMLE", "MIROC6", "MPI_ESM", "ACCESS", "EC_Earth3", "IPSL_CM6A", "CESM2", "CanESM5"]

intervals = np.arange(-1.0, 1.1, 0.1)

cmap = mcolors.ListedColormap(palettable.cmocean.diverging.Balance_20.mpl_colors)
extend = 'both'
panel_fs = 10
row_label_fs = 8
corr_fs = 9
cbar_fs = 8
# set_science_advances_style(
fig = plt.figure(figsize=(25, 30)) 
gs = gridspec.GridSpec(9, 4, figure=fig, width_ratios=[1,1,1,1], wspace=0.08, hspace=0.015)

# Create a 8x3 grid of subplots
axes = {}
for i, var in enumerate(variable_name):
    for j, period in enumerate(periods):
        is_left = j == 0
        is_bottom_row = i >= 8
        
        ax = plt.subplot(gs[i, j], projection=ccrs.Robinson(180))
        ax.set_global()
        axes[i, j] = ax
        if j == 0:
            # Add cyclic points
            trend_data = trend_2013_2022[variable_name[i]]
            p_values = p_value_2013_2022[variable_name[i]]
            trend_with_cyclic, lons_cyclic = cutil.add_cyclic_point(trend_data, coord=lon)
            p_values_with_cyclic, lons_cyclic = cutil.add_cyclic_point(p_values, coord=lon)
            
            contour_obj = plot_trend_with_significance(trend_with_cyclic, lat, lons_cyclic, p_values_with_cyclic, 
                                                       levels=intervals, extend = extend,
                                                       cmap='RdBu_r', title=" ", ax=ax, 
                                                       show_xticks = is_bottom_row, show_yticks = is_left)
        elif j == 1:
            trend_data = trend_1993_2022[variable_name[i]]
            p_values = p_value_1993_2022[variable_name[i]]
            trend_with_cyclic, lons_cyclic = cutil.add_cyclic_point(trend_data, coord=lon)
            p_values_with_cyclic, lons_cyclic = cutil.add_cyclic_point(p_values, coord=lon)
            
            contour_obj1 = plot_trend_with_significance(trend_with_cyclic, lat, lons_cyclic, p_values_with_cyclic, 
                                                        levels=intervals, extend = extend,
                                                        cmap='RdBu_r', title=" ", ax=ax, 
                                                        show_xticks = is_bottom_row, show_yticks = is_left)
        elif j == 2:
            trend_data = trend_1979_2022[variable_name[i]]
            p_values = p_value_1979_2022[variable_name[i]]
            trend_with_cyclic, lons_cyclic = cutil.add_cyclic_point(trend_data, coord=lon)
            p_values_with_cyclic, lons_cyclic = cutil.add_cyclic_point(p_values, coord=lon)
            
            contour_obj2 = plot_trend_with_significance(trend_with_cyclic, lat, lons_cyclic, p_values_with_cyclic, 
                                                        levels=intervals, extend = extend,
                                                        cmap='RdBu_r', title=" ", ax=ax, 
                                                        show_xticks = is_bottom_row, show_yticks = is_left)
        else:
            trend_data = trend_1963_2022[variable_name[i]]
            p_values = p_value_1963_2022[variable_name[i]]
            trend_with_cyclic, lons_cyclic = cutil.add_cyclic_point(trend_data, coord=lon)
            p_values_with_cyclic, lons_cyclic = cutil.add_cyclic_point(p_values, coord=lon)
            
            contour_obj2 = plot_trend_with_significance(trend_with_cyclic, lat, lons_cyclic, p_values_with_cyclic, 
                                                        levels=intervals, extend = extend,
                                                        cmap='RdBu_r', title=" ", ax=ax, 
                                                        show_xticks = is_bottom_row, show_yticks = is_left)

# add the title for each row
for i, var in enumerate(variable_name):
    axes[i, 0].text(-0.2, 0.5, titles_rows[i], va='center', ha='center', rotation=90, 
                    fontsize=20,transform=axes[i, 0].transAxes)
    axes[i, 0].text(-0.08, 1.05, rows_label[i], va='bottom', ha='right', rotation='horizontal',
                    fontsize=24, fontweight='bold', transform=axes[i, 0].transAxes)

# add the title for each column
for j in range(4):
    axes[0,j].text(0.5, 1.05, titles_columns[j], va='bottom', ha='center', rotation='horizontal', 
                   fontsize=20,transform=axes[0, j].transAxes)

# Add the pattern correlation text to the second to last rows of each column subplot
for i in np.arange(1,9,1):
    axes[i, 0].text(0.65, 1.0, f"corr: {trend_pattern_correlation_10yr[i-1]:.2f}", va='bottom', ha='left', fontsize=20,transform=axes[i, 0].transAxes)
    axes[i, 1].text(0.65, 1.0, f"corr: {trend_pattern_correlation_30yr[i-1]:.2f}", va='bottom', ha='left', fontsize=20,transform=axes[i, 1].transAxes)
    axes[i, 2].text(0.65, 1.0, f"corr: {trend_pattern_correlation_44yr[i-1]:.2f}", va='bottom', ha='left', fontsize=20,transform=axes[i, 2].transAxes)
    axes[i, 3].text(0.65, 1.0, f"corr: {trend_pattern_correlation_60yr[i-1]:.2f}", va='bottom', ha='left', fontsize=20,transform=axes[i, 3].transAxes)

# Add horizontal colorbars
cbar_ax = fig.add_axes([0.25, 0.08, 0.5, 0.01])
cbar = plt.colorbar(contour_obj, cax=cbar_ax, orientation='horizontal', extend=extend)
cbar.ax.tick_params(labelsize=16)
cbar.set_label('Annual SAT trend (°C per decade)', fontsize=18)
cbar.ax.tick_params(direction='out', length=8, width=2)  
# remove the minor ticks from the bar labels
cbar.ax.minorticks_off()

# plt.figure(constrained_layout=True)
figure_output = '/work/mh0033/m301036/OBS_LPS_revision/docs/Figs/FIG_S7_S8/'
for ext in ("png", "pdf",):
    fig.savefig(figure_output+f"FIGURE_S7_OBS_LE_forced_trend_comparison_Satellite.{ext}", format=ext, dpi=300, bbox_inches='tight')

plt.show()